<a href="https://colab.research.google.com/github/wisidorio/colab-notebooks/blob/main/01_setup_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Preparar o conjunto de dados na sessão do Colab

Baixa nada: lê o `.tar.gz` que já está no Google Drive, **confere a
integridade**, extrai para o disco local da sessão (`/content`) e **verifica
o que saiu** antes de qualquer outra coisa.

Este notebook **não** faz pré-processamento nem treino — ele só deixa o
dataset pronto. Rode-o uma vez no começo de cada sessão (o `/content` é
apagado quando a sessão reinicia; o Drive não).

---

### Duas regras que valem a pena saber antes

**1. Extraia para `/content`, nunca para dentro do Drive.**
O `/content` é disco local (SSD) da máquina virtual. Ler ~28 mil arquivos
pequenos pelo *mount* do Drive durante o treino é muito mais lento, e o
Drive costuma limitar a taxa de requisições.

**2. Não copie a pasta extraída de volta para o Drive.**
As imagens do `merged_yolo/` são *hard links* para as do `merged/` — os dois
formatos dividem os mesmos bytes. O Drive não guarda hard links, então a
cópia vira ~2,9 GB de arquivos independentes em vez de 1,5 GB, e leva muito
tempo. O `.tar.gz` no Drive já é a cópia permanente; o resto é descartável e
se reconstrói em poucos minutos.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Configuração

Só esta célula muda se a data do pacote mudar.

In [ ]:
from pathlib import Path

WD       = Path('/content/drive/MyDrive/2026/tcc_mba/dbs')   # onde está o .tar.gz
STEM     = 'brands-merged-2026-09-22'                        # nome do pacote (sem .tar.gz)
DEST     = Path('/content/dbs')                               # disco local da sessão

ARCHIVE  = WD / f'{STEM}.tar.gz'
DS       = DEST / STEM                                       # raiz do dataset extraído
DATA_YAML = DS / 'merged_yolo' / 'data.yaml'

print('arquivo :', ARCHIVE)
print('destino :', DS)

arquivo : /content/drive/MyDrive/2026/tcc_mba/dbs/brands-merged-2026-09-22.tar.gz
destino : /content/dbs/brands-merged-2026-09-22


In [ ]:
import shutil

assert ARCHIVE.exists(), f'não encontrei {ARCHIVE} — confira o caminho em WD'
print(f'pacote  : {ARCHIVE.stat().st_size / (1 << 30):.2f} GiB')

livre = shutil.disk_usage('/content').free / (1 << 30)
print(f'livre   : {livre:.1f} GiB em /content')
assert livre > 4, 'espaço insuficiente em /content para extrair o pacote'

pacote  : 1.46 GiB
livre   : 205.5 GiB em /content


## 1. Integridade

`sha256sum -c` recalcula o hash do arquivo e compara com o que está no
`SHA256SUMS.txt`. O `assert` interrompe o notebook se não bater — é melhor
parar aqui do que treinar em cima de um arquivo corrompido no upload.

> Cuidado com o algoritmo: o arquivo de referência é **SHA-256**. Conferir
> com `sha1sum` gera um número que nunca vai coincidir com ele.

Leva cerca de um minuto (lê 1,5 GB do Drive).

In [ ]:
import subprocess

r = subprocess.run(['sha256sum', '-c', 'SHA256SUMS.txt'],
                   cwd=WD, capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip())
assert r.returncode == 0, 'INTEGRIDADE FALHOU — reenvie o pacote para o Drive'

brands-merged-2026-09-22.tar.gz: OK


## 2. Extração

`-C` manda o `tar` extrair no destino certo (sem isso ele extrai no
diretório atual e sobra uma cópia para mover depois). Sem `-v`: a listagem
verbosa são ~28 mil linhas que só incham o notebook.

Leva de um a três minutos. Se a pasta já existir, a célula não refaz o
trabalho — mude `FORCE` para `True` para forçar.

In [ ]:
FORCE = False

if DS.exists() and not FORCE:
    print(f'já extraído em {DS} (use FORCE=True para refazer)')
else:
    DEST.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['tar', '-xzf', str(ARCHIVE), '-C', str(DEST)],
                       capture_output=True, text=True)
    assert r.returncode == 0, f'tar falhou:\n{r.stderr[:2000]}'
    print(f'extraído em {DS}')

print()
print(*sorted(p.name for p in DS.iterdir()), sep='\n')

extraído em /content/dbs/brands-merged-2026-09-22

README.md
classes.txt
classes_train.txt
conjunto_mesclado.html
merged
merged_yolo


## 3. Verificação do que foi extraído

Contagens esperadas, conferidas contra os arquivos que chegaram. Qualquer
divergência interrompe o notebook com a lista do que está errado — em vez
de imprimir algo que passaria despercebido.

Também confere se os *hard links* sobreviveram: se sobreviveram, as duas
cópias da mesma imagem compartilham o `inode` e o dataset ocupa 1,6 GB em
vez de 2,9 GB.

In [ ]:
import yaml

ESPERADO = {'train': 6437, 'val': 1396, 'test': 1370}
N_IMAGENS, N_CLASSES, N_NEGATIVAS = 9203, 80, 1000
problemas = []

# --- pareamento imagem/rótulo e tamanho de cada divisão ---
for split, n in ESPERADO.items():
    imgs = {p.stem for p in (DS / 'merged_yolo' / 'images' / split).glob('*.jpg')}
    lbls = {p.stem for p in (DS / 'merged_yolo' / 'labels' / split).glob('*.txt')}
    if len(imgs) != n:
        problemas.append(f'{split}: {len(imgs)} imagens, esperado {n}')
    if imgs != lbls:
        problemas.append(f'{split}: {len(imgs ^ lbls)} arquivos sem par imagem/rótulo')

# --- mestre VOC ---
n_xml = len(list((DS / 'merged' / 'Annotations').glob('*.xml')))
n_jpg = len(list((DS / 'merged' / 'JPEGImages').glob('*.jpg')))
if (n_xml, n_jpg) != (N_IMAGENS, N_IMAGENS):
    problemas.append(f'VOC: {n_xml} xml / {n_jpg} jpg, esperado {N_IMAGENS} de cada')

# --- classes: a lista, o data.yaml e a ordem dos ids ---
classes = [c for c in (DS / 'classes_train.txt').read_text(encoding='utf-8').split('\n') if c]
if len(classes) != N_CLASSES:
    problemas.append(f'classes_train.txt tem {len(classes)} classes, esperado {N_CLASSES}')

data = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
if data['nc'] != len(classes):
    problemas.append(f"data.yaml nc={data['nc']} mas classes_train.txt tem {len(classes)}")
if [data['names'][i] for i in sorted(data['names'])] != classes:
    problemas.append('os nomes do data.yaml não batem com classes_train.txt na mesma ordem')

# --- negativas: rótulo vazio é de propósito (imagem de fundo) ---
vazios = sum(1 for s in ESPERADO
             for p in (DS / 'merged_yolo' / 'labels' / s).glob('*.txt')
             if p.stat().st_size == 0)
if vazios != N_NEGATIVAS:
    problemas.append(f'{vazios} rótulos vazios, esperado {N_NEGATIVAS}')

# --- os hard links sobreviveram à extração? ---
amostra = next(iter((DS / 'merged_yolo' / 'images' / 'train').glob('*.jpg')))
a = amostra.stat()
b = (DS / 'merged' / 'JPEGImages' / amostra.name).stat()
compartilham = (a.st_dev, a.st_ino) == (b.st_dev, b.st_ino)
if not compartilham:
    problemas.append('hard links não preservados — o dataset ocupa o dobro do necessário')

assert not problemas, 'PROBLEMAS NA EXTRAÇÃO:\n  - ' + '\n  - '.join(problemas)

print(f'ok: {N_IMAGENS} imagens, {N_CLASSES} classes, '
      + ', '.join(f'{s}={n}' for s, n in ESPERADO.items()))
print(f'    {vazios} rótulos vazios (negativas), hard links preservados: {compartilham}')

ok: 9203 imagens, 80 classes, train=6437, val=1396, test=1370
    1000 rótulos vazios (negativas), hard links preservados: True


## 4. Apontar o `data.yaml` para esta sessão

O `data.yaml` vem com o caminho absoluto da máquina onde o pacote foi
gerado. Só a linha `path:` muda — `train:`, `val:` e `test:` são relativos a
ela. O arquivo é deixado **exatamente como o pipeline gerou** dentro do
pacote justamente para que a conferência feita lá continue valendo; o ajuste
acontece aqui, na cópia da sessão.

In [ ]:
linhas = DATA_YAML.read_text(encoding='utf-8').splitlines()
novo = [f'path: {DS}/merged_yolo' if l.startswith('path:') else l for l in linhas]
assert any(l.startswith('path:') for l in novo), 'data.yaml sem a linha path:'
DATA_YAML.write_text('\n'.join(novo) + '\n', encoding='utf-8')

data = yaml.safe_load(DATA_YAML.read_text(encoding='utf-8'))
for chave in ('train', 'val', 'test'):
    caminho = Path(data['path']) / data[chave]
    assert caminho.is_dir(), f'data.yaml {chave} -> {caminho} não existe'

print('\n'.join(DATA_YAML.read_text(encoding='utf-8').splitlines()[:6]))
print('...')
print(f"\nok: os três caminhos do data.yaml existem (nc={data['nc']})")

# Merged Brazilian food/beverage logo set -- see docs/PIPELINE.md
path: /content/dbs/brands-merged-2026-09-22/merged_yolo
train: images/train
val: images/val
test: images/test

...

ok: os três caminhos do data.yaml existem (nc=80)


## 5. Pronto

O dataset está em `DS` e o `data.yaml` aponta para os arquivos desta sessão.
Use assim no notebook de treino:

```python
from ultralytics import YOLO
model = YOLO('yolo11n.pt')
model.train(data=str(DATA_YAML), epochs=100, imgsz=640)
```

### Se quiser liberar espaço no Drive

Se a pasta `brands-merged-2026-09-22/` já foi copiada para o Drive numa
sessão anterior, ela pode ser apagada: são ~2,9 GB que este notebook
reconstrói em poucos minutos a partir do `.tar.gz`. Confira o que vai sair
antes de apagar:

```python
# !du -sh "$WD/brands-merged-2026-09-22"
# !rm -rf "$WD/brands-merged-2026-09-22"
```

### O que documenta o conjunto

O `README.md` dentro do pacote traz a estrutura de pastas, os dois formatos
de anotação, as 80 classes com seus ids e contagens, e as armadilhas
conhecidas — entre elas: `classes.txt` e `classes_train.txt` têm 80 linhas
cada e **não são a mesma lista** (para treinar, use `classes_train.txt`), e
as classes `Perdigão` e `Mamma Dalva` existem **apenas no treino**, então não
espere métricas de avaliação para elas.

In [ ]:
print(Path(DS / 'README.md').read_text(encoding='utf-8')[:1200])